In [4]:
import pandas as pd
import numpy as np

In [5]:
train_trans = pd.read_csv("../data/raw/train_transaction.csv")
train_id = pd.read_csv("../data/raw/train_identity.csv")

test_trans = pd.read_csv("../data/raw/test_transaction.csv")
test_id = pd.read_csv("../data/raw/test_identity.csv")

In [6]:
train = train_trans.merge(train_id, on="TransactionID", how="left")
test = test_trans.merge(test_id, on="TransactionID", how="left")
y = train["isFraud"]
train.drop(columns=["isFraud"], inplace=True)

In [8]:
missing = train.isnull().mean()

drop_cols = missing[missing > 0.9].index

train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

In [9]:

train["TransactionAmt"] = np.log1p(train["TransactionAmt"])
test["TransactionAmt"] = np.log1p(test["TransactionAmt"])

In [10]:
train["hour"] = (train["TransactionDT"] // 3600) % 24
test["hour"] = (test["TransactionDT"] // 3600) % 24

train["day"] = (train["TransactionDT"] // (3600 * 24)) % 7
test["day"] = (test["TransactionDT"] // (3600 * 24)) % 7

C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\1272056876.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["hour"] = (train["TransactionDT"] // 3600) % 24
C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\1272056876.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["hour"] = (test["TransactionDT"] // 3600) % 24
C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\1272056876.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

In [11]:
train["amt_decimal"] = (
    (train["TransactionAmt"] - train["TransactionAmt"].astype(int)) * 1000
)

test["amt_decimal"] = (
    (test["TransactionAmt"] - test["TransactionAmt"].astype(int)) * 1000
)

C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\3974533811.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["amt_decimal"] = (
C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\3974533811.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["amt_decimal"] = (


In [12]:
freq = train["card1"].value_counts()

train["card1_freq"] = train["card1"].map(freq)
test["card1_freq"] = test["card1"].map(freq)

C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\553681112.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["card1_freq"] = train["card1"].map(freq)
C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\553681112.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["card1_freq"] = test["card1"].map(freq)


In [13]:
common_cols = train.columns.intersection(test.columns)

for col in common_cols:

    # Missing indicator
    train[col + "_is_missing"] = train[col].isnull().astype(int)
    test[col + "_is_missing"] = test[col].isnull().astype(int)

    # Fill numeric values
    if train[col].dtype in ["float64", "int64"]:
        train[col].fillna(train[col].median(), inplace=True)
        test[col].fillna(test[col].median(), inplace=True)

    # Fill categorical values
    else:
        train[col].fillna("missing", inplace=True)
        test[col].fillna("missing", inplace=True)

C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\118049729.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[col + "_is_missing"] = train[col].isnull().astype(int)
C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\118049729.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test[col + "_is_missing"] = test[col].isnull().astype(int)
C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\118049729.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inpl

In [14]:
cat_cols = train.select_dtypes(include=["object"]).columns
cat_cols = train.columns.intersection(test.columns).intersection(cat_cols)

for col in cat_cols:
    freq = train[col].value_counts()

    train[col] = train[col].map(freq)
    test[col] = test[col].map(freq)

    test[col] = test[col].fillna(0)

C:\Users\m.1032\AppData\Local\Temp\ipykernel_21368\3893403212.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train.select_dtypes(include=["object"]).columns


In [15]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [16]:
print("Preprocessing Done!")
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("end")

Preprocessing Done!
Train shape: (590540, 822)
Test shape: (506691, 822)
